# 04_206 · Qwen en cascada y jerárquico multitarea · cuatro daños

Compara el Qwen plano de `04_205` con dos estructuras construidas sobre su adaptador terminado: una cascada logística y una cabeza neuronal jerárquica multitarea. Este cuaderno **no debe ejecutarse hasta que `04_205` termine**. No modifica su adaptador ni realiza un segundo fine-tuning end-to-end.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'cuadernos': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from scripts_auxiliares import experimentos_qwen_jerarquico_4 as qh
from scripts_auxiliares import entrenar_qwen_acoso_amenaza as q4
print('Estado 04_205:', q4.resume_status())
print('Objetivos:', qh.TARGET_LABELS)

## 1. Contrato y requisito

La siguiente celda se detiene deliberadamente si `04_205` no generó `finetuning.json` y `best_adapter`. Cuando esté listo, verifica que Qwen y los experimentos jerárquicos compartan el mismo hash de dataset y los mismos splits.

In [ ]:
USE_EXPANDED_SAFE = True
context = qh.load_context(use_expanded_safe=USE_EXPANDED_SAFE)
display(context['qwen'])
display(context['expansion'])
assert context['qwen_audit']['dataset_sha256'] == context['frozen']['dataset_sha256']
assert context['expansion']['validation_or_test_videos_used'] is False

## 2. Representaciones Qwen congeladas

Se guardan los 21 logits de `04_205`: cuatro operativos, 14 finos auxiliares y tres flags. Son representaciones generadas únicamente a partir del texto; las etiquetas no son entradas. El cache registra hash del adaptador y de los IDs.

Con `USE_EXPANDED_SAFE=True`, Qwen debe inferir también los `SEGURO` adicionales permitidos. Esto puede tardar bastante en CPU, pero se realiza una sola vez y luego se reutiliza. Aunque el corpus tiene más de 110 mil `SEGURO`, train usa sólo aquellos cuyos videos no aparecen en validation/test.

In [ ]:
FORCE_FEATURES = False
features = qh.extract_features(context, force=FORCE_FEATURES)
display({split: values.shape for split, values in features.items()})

## 3. Dos diseños Qwen jerárquicos

La cascada entrena una puerta binaria con todos los negativos permitidos y cuatro cabezas condicionales con daños y negativos difíciles. El multitarea usa una capa compartida de 32 unidades, cabeza binaria y cuatro cabezas temáticas. La pérdida temática se enmascara en `SEGURO` adicional; sólo la cabeza binaria aprovecha esos chunks.

Ambos modelos fijan umbrales en validation y se comparan con las probabilidades calibradas del Qwen plano sobre exactamente el mismo test. La inferencia estadística usa bootstrap pareado por videos.

In [ ]:
FORCE = False
BOOTSTRAP_REPLICATES = 1_000
result = qh.run_experiment(
    force=FORCE,
    use_expanded_safe=USE_EXPANDED_SAFE,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
)
display(result['selection'])
display(result['paired_decisions_vs_qwen_flat'])

In [ ]:
comparison = pd.read_csv(qh.METRICS_DIR / 'comparacion.csv')
display(comparison)
display(pd.DataFrame(result['models']['qwen_frozen_joint']['training']['history']))
display(Image(filename=str(qh.FIGURES_DIR / 'comparacion_test.png')))

In [ ]:
display(Markdown(
    f'**Resultado:** `{qh.RESULT_PATH.relative_to(ROOT)}`  \n'
    f'**Informe:** `{qh.REPORT_PATH.relative_to(ROOT)}`  \n'
    f'**Modelos:** `{qh.MODEL_DIR.relative_to(ROOT)}`'
))

## Referencias (APA 7)

Cawley, G. C., & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079–2107. https://www.jmlr.org/papers/v11/cawley10a.html

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall/CRC.

Zhou, J., Ma, C., Long, D., Xu, G., Ding, N., Zhang, H., Xie, P., & Liu, G. (2020). Hierarchy-aware global model for hierarchical text classification. In *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics* (pp. 1106–1117). Association for Computational Linguistics. https://doi.org/10.18653/v1/2020.acl-main.104